# BSDT Sonar — H7 + H8a Combined Pipeline

**Single Colab run. Est. time: ~2.5 hours on A100/RTX.**

```
Phase A  H7 Fast   — find best annealing schedule (4 × n=750,1000)
Phase B  H8a       — Nullspace sub-problem solver on failures
Phase C  WalkSAT   — final cleanup
```

**The core idea (H8a):**  
After annealing, failing particles sit at *wrong corners* with ~27 clause violations.  
Those violations touch ≤ 3×27 = **81 variables** out of 1000.  
Freeze the other 919 (already correct), re-anneal only the 81-variable sub-problem.  
Sub-problem is always inside the 100% solve zone → polynomial total cost.

Author: Odeyemi Olusegun Israel, Independent Researcher, Derby UK

In [ ]:
import torch, numpy as np, time, matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CORE ENGINE  (freeze_mask support for nullspace sub-solving)
# ════════════════════════════════════════════════════════════════

class BSDTSonarEngine:
    def __init__(self, n, num_instances=50, num_particles=1000,
                 alpha=3.0, mu_scale=0.1, device=device):
        self.n=n; self.ni=num_instances; self.np_=num_particles
        self.alpha=alpha; self.mu_scale=mu_scale; self.device=device
        self.m=int(alpha*n)

    def generate_instances(self):
        ni,m,n = self.ni,self.m,self.n
        cv = torch.zeros(ni,m,3,dtype=torch.long,device=self.device)
        cs = torch.zeros(ni,m,3,dtype=torch.float,device=self.device)
        for i in range(ni):
            for c in range(m):
                cv[i,c] = torch.randperm(n,device=self.device)[:3]
                cs[i,c] = torch.randint(0,2,(3,),device=self.device).float()*2-1
        return cv,cs

    def compute_mu(self,cv):
        deg = torch.zeros(self.ni,self.n,device=self.device)
        for p in range(3):
            deg.scatter_add_(1,cv[:,:,p],torch.ones(self.ni,self.m,device=self.device))
        lmax = 0.25*deg.max(dim=1).values
        return (self.mu_scale*lmax).clamp(min=0.01), lmax

    def energy_and_grad(self,s,cv,cs,mu):
        ni,np_,m,n = self.ni,s.shape[1],self.m,self.n
        cv4  = cv.unsqueeze(1).expand(ni,np_,m,3)
        s_at = torch.gather(s.unsqueeze(2).expand(ni,np_,m,n),3,cv4)
        cs4  = cs.unsqueeze(1).expand(ni,np_,m,3)
        lit  = (1.0-cs4*s_at)/2.0
        l0,l1,l2 = lit[...,0],lit[...,1],lit[...,2]
        Ec   = (l0*l1*l2).sum(dim=2)
        mu3  = mu.view(ni,1,1)
        g    = torch.zeros(ni,np_,n,device=self.device)
        for pos,dl in enumerate([
            (-cs4[...,0]/2)*l1*l2,
             l0*(-cs4[...,1]/2)*l2,
             l0*l1*(-cs4[...,2]/2)]):
            g.scatter_add_(2,cv[:,: ,pos].unsqueeze(1).expand(ni,np_,m),dl)
        g = g + mu3*(-4.0*s*(1.0-s**2))
        return Ec+(mu3*(1.0-s**2)**2).sum(dim=2), Ec, g

    def gradient_flow(self,s,cv,cs,mu,steps,dt=0.05,beta=0.90,
                      mu_override=None,freeze_mask=None):
        """
        freeze_mask : (ni,n) bool — frozen vars get zero gradient & stay fixed.
        Enables nullspace sub-problem solving without re-indexing.
        """
        ni,np_,n = self.ni,s.shape[1],self.n
        v     = torch.zeros_like(s)
        plat  = torch.zeros(ni,np_,device=self.device)
        bE    = torch.full((ni,np_),float('inf'),device=self.device)
        fm    = freeze_mask.unsqueeze(1).float() if freeze_mask is not None else None

        for step in range(steps):
            mu_eff = mu_override(step,steps,mu) if mu_override else mu
            _,Ec,g = self.energy_and_grad(s,cv,cs,mu_eff)

            if fm is not None: g = g*(1.0-fm)  # zero grad on frozen vars

            imp  = Ec<bE; bE = torch.where(imp,Ec,bE)
            plat = torch.where(imp,torch.zeros_like(plat),plat+1)
            pm   = plat>=50
            dec  = 1.0/(1.0+0.002*step)
            dte  = dt*dec/(1.0+0.05*g.norm(dim=2,keepdim=True).clamp(1e-10))
            dte  = dte*torch.where(pm.unsqueeze(2),
                                    torch.full_like(dte,2.0),torch.ones_like(dte))
            gam  = (Ec.clamp(0)/(Ec.clamp(0)+1.0)).unsqueeze(2)
            v    = beta*v - dte*(1.0+gam)*g
            ns   = torch.where(pm.unsqueeze(2),
                               torch.full_like(v,0.03*dec*4),
                               torch.full_like(v,0.03*dec))
            sn   = torch.clamp(s+v+torch.randn_like(s)*ns,-1.0,1.0)
            if fm is not None:                   # keep frozen vars fixed
                sn = torch.where(fm.bool(),s,sn)
                v  = torch.where(fm.bool(),torch.zeros_like(v),v)
            s = sn
        return s,bE

    def solve_rate(self,s,cv,cs,mu):
        sr = torch.sign(s+1e-10)
        _,E,_ = self.energy_and_grad(sr,cv,cs,mu)
        best  = E.min(dim=1).values
        rate  = (best<0.5).float().mean().item()
        return rate, np.sqrt(rate*(1-rate)/self.ni), best

    def per_clause_energy(self,s_best,cv,cs):
        """s_best:(ni,n) → per-clause energies (ni,m)"""
        ni,m,n = self.ni,self.m,self.n
        cv4  = cv.unsqueeze(1).expand(ni,1,m,3)
        s_ex = s_best.unsqueeze(1).unsqueeze(2).expand(ni,1,m,n)
        s_at = torch.gather(s_ex,3,cv4)
        cs4  = cs.unsqueeze(1).expand(ni,1,m,3)
        lit  = (1.0-cs4*s_at)/2.0
        return (lit[...,0]*lit[...,1]*lit[...,2]).squeeze(1)  # (ni,m)

print('BSDTSonarEngine loaded.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# SCHEDULES
# ════════════════════════════════════════════════════════════════

def sched_cosine(t):   return (1.0-np.cos(np.pi*t))/2.0
def sched_power10(t):  return t**10
def sched_power20(t):  return t**20
def sched_delay70(t):
    if t<0.7: return 0.0
    t2=(t-0.7)/0.3; return (1.0-np.cos(np.pi*t2))/2.0

SCHEDULES = {
    'cosine':  sched_cosine,
    'power10': sched_power10,
    'power20': sched_power20,
    'delay70': sched_delay70,
}

def make_mu_override(fn):
    def mu_ov(step,total,mu_b,_fn=fn):
        return mu_b*_fn(step/max(total-1,1))
    return mu_ov

t_arr = np.linspace(0,1,1000)
print(f'  {"Schedule":>10} | μ/2   μ/4   μ/10')
print('  '+'-'*36)
for name,fn in SCHEDULES.items():
    v=np.array([fn(t) for t in t_arr])
    print(f'  {name:>10} | {(v<0.5).mean():.0%}   {(v<0.25).mean():.0%}   {(v<0.1).mean():.0%}')
print()
print('WalkSAT and Nullspace solvers defined below.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# STAGE 2 — NULLSPACE SUB-PROBLEM SOLVER
# ════════════════════════════════════════════════════════════════

def nullspace_subsolve(s1, cv, cs, mu, engine, mu_override,
                       num_particles_sub=1000):
    """
    For each failing instance:
      1. Find violated clauses → V* (variables in violated clauses)
      2. Freeze V_fixed = all vars NOT in V*
      3. Re-anneal with freeze_mask on V_fixed
      4. Sub-problem steps ∝ sqrt(|V*|), not sqrt(n)
    """
    ni,n = engine.ni, engine.n

    # Identify failures
    s_round = torch.sign(s1+1e-10)
    _,E,_ = engine.energy_and_grad(s_round,cv,cs,mu)
    best_E    = E.min(dim=1).values
    fail_mask = best_E >= 0.5
    n_fail    = fail_mask.sum().item()

    if n_fail == 0:
        print('    Nullspace: no failures — skipped')
        return s1, {'recovered':0,'n_fail':0,'vstar_mean':0,'vstar_max':0,'elapsed':0}

    # Best particle per instance
    best_idx = E.argmin(dim=1)
    s_best   = s_round[torch.arange(ni,device=engine.device), best_idx]  # (ni,n)

    # Per-clause energies to find V*
    Epc = engine.per_clause_energy(s_best, cv, cs)  # (ni,m)

    freeze_mask = torch.ones(ni,n,dtype=torch.bool,device=engine.device)
    vstar_sizes = []
    for inst in range(ni):
        if not fail_mask[inst]: continue
        viol_clauses = (Epc[inst] > 0.1).nonzero(as_tuple=True)[0]
        vstar = cv[inst][viol_clauses].reshape(-1).unique()
        freeze_mask[inst, vstar] = False   # V* is NOT frozen
        vstar_sizes.append(len(vstar))

    vstar_arr = np.array(vstar_sizes) if vstar_sizes else np.array([0])
    vstar_max = int(vstar_arr.max())
    sub_steps = min(int(500*np.sqrt(max(vstar_max,10))), 5000)

    print(f'    Nullspace: {n_fail} failures  |V*| mean={vstar_arr.mean():.1f} '
          f'max={vstar_max}  ratio={vstar_arr.mean()/n:.3f}  sub_steps={sub_steps}')

    # Init sub-particles: copy best, add noise only on V* vars
    s_sub = s_best.unsqueeze(1).expand(ni,num_particles_sub,n).clone()
    vmask = (~freeze_mask).unsqueeze(1).expand(ni,num_particles_sub,n)
    s_sub = torch.where(vmask,
                        torch.clamp(s_sub+torch.randn_like(s_sub)*0.5,-1.0,1.0),
                        s_sub)

    t0 = time.time()
    s_sub_out,_ = engine.gradient_flow(
        s_sub,cv,cs,mu,sub_steps,dt=0.05,
        mu_override=mu_override,freeze_mask=freeze_mask)
    elapsed = time.time()-t0

    # Best sub-particle per instance
    s_sr = torch.sign(s_sub_out+1e-10)
    _,E_sub,_ = engine.energy_and_grad(s_sr,cv,cs,mu)
    best_sub = E_sub.argmin(dim=1)
    s_sub_best = s_sr[torch.arange(ni,device=engine.device),best_sub]  # (ni,n)

    # Count recoveries
    _,E2,_ = engine.energy_and_grad(s_sub_best.unsqueeze(1),cv,cs,mu)
    sub_solved = E2.squeeze(1) < 0.5
    fail_idx   = fail_mask.nonzero(as_tuple=True)[0]
    recovered  = sub_solved[fail_idx].sum().item()

    # Merge: replace slot-0 of failing instances with sub-solve best
    s_out = s1.clone()
    s_out[fail_idx, 0] = s_sub_best[fail_idx]

    print(f'    Nullspace done {elapsed:.0f}s — recovered {recovered}/{n_fail}')
    return s_out, {'recovered':recovered,'n_fail':n_fail,
                   'vstar_mean':float(vstar_arr.mean()),
                   'vstar_max':vstar_max,'elapsed':elapsed}


# ════════════════════════════════════════════════════════════════
# STAGE 3 — WALKSAT
# ════════════════════════════════════════════════════════════════

def walksat_single(cv_np, cs_np, init, n, max_flips=50000, p=0.57):
    m = cv_np.shape[0]; s = init.copy()
    def sat(c):
        i,j,k=int(cv_np[c,0]),int(cv_np[c,1]),int(cv_np[c,2])
        return ((1-cs_np[c,0]*s[i])/2*(1-cs_np[c,1]*s[j])/2*
                (1-cs_np[c,2]*s[k])/2) < 0.5
    for _ in range(max_flips):
        un = [c for c in range(m) if not sat(c)]
        if not un: return s,True
        c  = un[np.random.randint(len(un))]
        vs = [int(cv_np[c,p2]) for p2 in range(3)]
        if np.random.random()<p:
            v=vs[np.random.randint(3)]
        else:
            best_v,bst=vs[0],m+1
            for v2 in vs:
                s[v2]=-s[v2]; cnt=sum(1 for c2 in range(m) if not sat(c2)); s[v2]=-s[v2]
                if cnt<bst: bst,best_v=cnt,v2
            v=best_v
        s[v]=-s[v]
    return s,False

def walksat_stage(s2,cv,cs,mu,engine):
    ni=engine.ni
    sr=torch.sign(s2+1e-10); _,E,_= engine.energy_and_grad(sr,cv,cs,mu)
    bE=E.min(dim=1).values; fail=bE>=0.5; bidx=E.argmin(dim=1)
    n_fail=fail.sum().item()
    if n_fail==0: return s2,0
    print(f'    WalkSAT: {n_fail} remaining...')
    cv_np=cv.cpu().numpy(); cs_np=cs.cpu().numpy(); s_np=sr.cpu().numpy()
    s_out=s2.clone(); rec=0; t0=time.time()
    for inst in range(ni):
        if not fail[inst]: continue
        res,ok=walksat_single(cv_np[inst],cs_np[inst],
                               s_np[inst,bidx[inst].item()],engine.n)
        if ok:
            rec+=1
            s_out[inst,0]=torch.tensor(res,dtype=torch.float,device=engine.device)
    print(f'    WalkSAT done {time.time()-t0:.0f}s — recovered {rec}/{n_fail}')
    return s_out,rec

print('Nullspace + WalkSAT solvers loaded.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE A — H7 FAST: find best schedule at n=750, 1000
# 4 schedules × 2 sizes × 50 instances × 1000 particles
# Est: ~60 min
# ════════════════════════════════════════════════════════════════

torch.manual_seed(42); np.random.seed(42)

print('='*60)
print('PHASE A — H7 FAST: Schedule Comparison')
print('Skipping n=500 (always 100%). 50 instances, 1000 particles.')
print('='*60)

NI_A, NP_A = 50, 1000
h7_results = {}

for n in [750, 1000]:
    steps  = min(int(500*np.sqrt(n)), 20000)
    engine = BSDTSonarEngine(n=n,num_instances=NI_A,
                              num_particles=NP_A,device=device)
    cv,cs  = engine.generate_instances()
    mu,_   = engine.compute_mu(cv)
    print(f'\n{"—"*60}')
    print(f'n={n}  m={engine.m}  steps={steps}')
    h7_results[n] = {}

    for name,fn in SCHEDULES.items():
        if device.type=='cuda': torch.cuda.empty_cache()
        s0 = torch.clamp(torch.randn(NI_A,NP_A,n,device=device)*0.3,-0.9,0.9)
        t0 = time.time()
        sf,_ = engine.gradient_flow(s0,cv,cs,mu,steps,
                                     mu_override=make_mu_override(fn))
        r,se,_ = engine.solve_rate(sf,cv,cs,mu)
        elapsed = time.time()-t0
        h7_results[n][name]={'rate':r,'se':se,'time':elapsed,'sf':sf.detach()}
        star='★' if r>=0.99 else '◆' if r>=0.95 else ' '
        print(f'  {star} {name:>10}: {r:6.1%} ± {se:.1%}  ({elapsed:.0f}s)')

# Pick best schedule
best_sched = max(SCHEDULES, key=lambda k: h7_results[1000][k]['rate'])
best_fn    = SCHEDULES[best_sched]
print(f'\n  ★ Best schedule at n=1000: {best_sched} '
      f'({h7_results[1000][best_sched]["rate"]:.1%})')
cosine_rate = h7_results[1000]['cosine']['rate']
uplift_A = h7_results[1000][best_sched]['rate'] - cosine_rate
print(f'    vs cosine ({cosine_rate:.1%}) → uplift = {uplift_A:+.1%}')
if abs(uplift_A) < 0.02:
    print('    = Schedule not the bottleneck — cosine baseline kept for H8a')
    best_sched, best_fn = 'cosine', sched_cosine

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE B — H8a: FULL PIPELINE with best schedule
# Stage 1: best annealing  → Stage 2: Nullspace  → Stage 3: WalkSAT
# 100 instances, 1000 particles
# Est: ~90 min
# ════════════════════════════════════════════════════════════════

torch.manual_seed(99); np.random.seed(99)

print('='*60)
print(f'PHASE B — H8a FULL PIPELINE (schedule={best_sched})')
print('100 instances, 1000 particles')
print('='*60)

NI_B, NP_B = 100, 1000
mu_ov_best = make_mu_override(best_fn)
h8_results = {}

for n in [500, 750, 1000]:
    steps  = min(int(500*np.sqrt(n)), 20000)
    engine = BSDTSonarEngine(n=n,num_instances=NI_B,
                              num_particles=NP_B,device=device)
    cv,cs  = engine.generate_instances()
    mu,_   = engine.compute_mu(cv)

    print(f'\n{"="*60}')
    print(f'n={n}  m={engine.m}  steps={steps}')
    print(f'{"="*60}')

    # ── Stage 1: annealing ───────────────────────────────
    s0 = torch.clamp(torch.randn(NI_B,NP_B,n,device=device)*0.3,-0.9,0.9)
    t0 = time.time()
    s1,_ = engine.gradient_flow(s0,cv,cs,mu,steps,mu_override=mu_ov_best)
    t1   = time.time()-t0
    r1,se1,bE1 = engine.solve_rate(s1,cv,cs,mu)
    print(f'  Stage 1 ({best_sched:>8}):  {r1:6.1%} ± {se1:.1%}  ({t1:.0f}s)')
    print(f'  Mean violations in failures: '
          f'{bE1[bE1>=0.5].mean().item():.1f}' if (bE1>=0.5).any() else
          '  No failures.')

    # ── Stage 2: nullspace sub-solver ────────────────────
    t0 = time.time()
    s2,stats2 = nullspace_subsolve(s1,cv,cs,mu,engine,
                                    mu_ov_best,NP_B)
    t2 = time.time()-t0
    r2,se2,_ = engine.solve_rate(s2,cv,cs,mu)
    print(f'  Stage 2 (nullspace):   {r2:6.1%} ± {se2:.1%}  '
          f'(+{r2-r1:+.1%}, {t2:.0f}s)')

    # ── Stage 3: WalkSAT ─────────────────────────────────
    t0 = time.time()
    s3,rec3 = walksat_stage(s2,cv,cs,mu,engine)
    t3 = time.time()-t0
    r3,se3,_ = engine.solve_rate(s3,cv,cs,mu)
    print(f'  Stage 3 (WalkSAT):    {r3:6.1%} ± {se3:.1%}  '
          f'(+{r3-r2:+.1%}, {t3:.0f}s)')

    total_t = t1+t2+t3
    print(f'  ────────────────────────────────────────')
    print(f'  FINAL n={n}:           {r3:6.1%}   total={total_t:.0f}s')
    if stats2['vstar_mean']>0:
        print(f'  Sub-problem ratio |V*|/n = '
              f'{stats2["vstar_mean"]/n:.3f} ({stats2["vstar_mean"]:.0f}/{n})')

    h8_results[n] = {
        'r1':r1,'r2':r2,'r3':r3,
        'vstar_mean':stats2['vstar_mean'],
        'vstar_max':stats2['vstar_max'],
        't1':t1,'t2':t2,'t3':t3,'total':total_t
    }

In [ ]:
# ════════════════════════════════════════════════════════════════
# RESULTS + CHARTS
# ════════════════════════════════════════════════════════════════

# H6 baselines for comparison
H6_p1   = {200:1.00,300:1.00,400:1.00,500:1.00,750:0.91,1000:0.52}
H6_final= {200:1.00,300:1.00,400:1.00,500:1.00,750:0.96,1000:0.76}

print('='*62)
print('FULL RESULTS SUMMARY')
print('='*62)
print(f'  {"n":>5} | {"S1 Anneal":>10} | {"S2 Nullsp":>10} | '
      f'{"S3 Walk":>9} | {"H6 Final":>9} | {"Δ H6":>7} | {"V*/n":>6}')
print('  '+'-'*70)
for n in [500,750,1000]:
    r = h8_results[n]
    h6f = H6_final.get(n,0)
    ratio = f"{r['vstar_mean']/n:.3f}" if r['vstar_mean']>0 else '  —  '
    delta = r['r3']-h6f
    flag = '★' if delta>0.05 else '◆' if delta>0.01 else '='
    print(f'  {n:>5} | {r["r1"]:>9.1%} | {r["r2"]:>9.1%} | '
          f'{r["r3"]:>8.1%} | {h6f:>8.1%} | {delta:>+6.1%} {flag} | {ratio:>6}')

print()
print('  POLYNOMIAL COMPLEXITY CHECK:')
print('  If |V*|/n stays constant or shrinks → Stage 2 is O(n)')
for n in [500,750,1000]:
    r=h8_results[n]
    if r['vstar_mean']>0:
        print(f'    n={n}: |V*|={r["vstar_mean"]:.0f}  ratio={r["vstar_mean"]/n:.4f}')


# ── Chart ────────────────────────────────────────────────────
fig = plt.figure(figsize=(16,10))
gs  = GridSpec(2,3,fig,hspace=0.35,wspace=0.3)

ns_b = [n for n in [500,750,1000] if n in h8_results]
r1s  = [h8_results[n]['r1'] for n in ns_b]
r2s  = [h8_results[n]['r2'] for n in ns_b]
r3s  = [h8_results[n]['r3'] for n in ns_b]
h6f  = [H6_final.get(n,0) for n in ns_b]
h6p1 = [H6_p1.get(n,0)    for n in ns_b]

# 1. Stage-by-stage solve rates
ax1 = fig.add_subplot(gs[0,0:2])
ax1.plot(ns_b,h6p1,'k--o',lw=1.5,ms=7,label='H6 Phase1 (cosine)')
ax1.plot(ns_b,h6f, 'b--s',lw=1.5,ms=7,label='H6 Final (+WalkSAT)')
ax1.plot(ns_b,r1s, 'g:^', lw=1.5,ms=7,label=f'H8a Stage1 ({best_sched})')
ax1.plot(ns_b,r2s, 'm-D', lw=2.0,ms=8,label='H8a Stage2 (+Nullspace)')
ax1.plot(ns_b,r3s, 'r-o', lw=2.5,ms=9,label='H8a Stage3 (+WalkSAT)',zorder=5)
ax1.fill_between(ns_b,h6f,r3s,alpha=0.15,color='red',label='H8a uplift over H6')
ax1.set_ylim(0.4,1.05); ax1.set_xlabel('n (variables)',fontsize=11)
ax1.set_ylabel('Solve Rate',fontsize=11)
ax1.set_title('H8a Pipeline vs H6 Baseline',fontsize=12,fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(True,alpha=0.3)

# 2. Schedule shapes
ax2 = fig.add_subplot(gs[0,2])
t_pl = np.linspace(0,1,500)
clrs = ['blue','orange','red','green']
for (nm,fn),col in zip(SCHEDULES.items(),clrs):
    v=np.array([fn(t) for t in t_arr])
    ax2.plot(t_pl,[fn(t) for t in t_pl],color=col,lw=2,
            label=f'{nm} ({(v<0.1).mean():.0%}@μ/10)')
ax2.axhline(0.1,color='k',ls=':',alpha=0.4); ax2.axhline(0.5,color='k',ls='--',alpha=0.4)
ax2.set_title('Schedule Shapes',fontsize=11); ax2.legend(fontsize=8)
ax2.set_xlabel('t'); ax2.set_ylabel('μ(t)/μ_target'); ax2.grid(True,alpha=0.3)

# 3. H7 schedule comparison
ax3 = fig.add_subplot(gs[1,0])
ns_a = [n for n in [750,1000] if n in h7_results]
for (nm,fn),col in zip(SCHEDULES.items(),clrs):
    rs = [h7_results[n][nm]['rate']*100 for n in ns_a]
    ax3.plot(ns_a,rs,'o-',color=col,lw=2,ms=8,label=nm)
ax3.set_title('H7: Schedule Comparison',fontsize=11)
ax3.set_xlabel('n'); ax3.set_ylabel('Solve Rate (%)'); ax3.legend(fontsize=8)
ax3.set_xticks([750,1000]); ax3.set_ylim(40,105); ax3.grid(True,alpha=0.3)

# 4. V* sub-problem size
ax4 = fig.add_subplot(gs[1,1])
ns_v = [n for n in ns_b if h8_results[n]['vstar_mean']>0]
if ns_v:
    vm   = [h8_results[n]['vstar_mean'] for n in ns_v]
    vrat = [v/n*100 for v,n in zip(vm,ns_v)]
    bars = ax4.bar(ns_v,vrat,color='steelblue',alpha=0.8,width=60)
    for bar,v in zip(bars,vm):
        ax4.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,
                 f'|V*|≈{v:.0f}',ha='center',fontsize=10,fontweight='bold')
ax4.axhline(10,color='red',ls='--',alpha=0.5,label='10% line')
ax4.set_title('Sub-Problem Size |V*|/n',fontsize=11)
ax4.set_xlabel('n'); ax4.set_ylabel('|V*|/n (%)'); ax4.legend(fontsize=9)
ax4.grid(True,alpha=0.3,axis='y')

# 5. Per-stage uplift at n=1000
ax5 = fig.add_subplot(gs[1,2])
if 1000 in h8_results:
    r=h8_results[1000]
    stgs = ['H6\nPhase1','H6\nFinal','H8a\nStage1','H8a\nStage2','H8a\nFinal']
    vals = [H6_p1[1000],H6_final[1000],r['r1'],r['r2'],r['r3']]
    cols = ['grey','steelblue','green','purple','red']
    ax5.bar(stgs,[v*100 for v in vals],color=cols,alpha=0.85,width=0.5)
    for i,(v,c) in enumerate(zip(vals,cols)):
        ax5.text(i,v*100+0.5,f'{v:.0%}',ha='center',fontsize=10,fontweight='bold')
ax5.set_title('n=1000: Stage-by-Stage Comparison',fontsize=11)
ax5.set_ylabel('Solve Rate (%)'); ax5.set_ylim(0,108)
ax5.grid(True,alpha=0.3,axis='y')

plt.suptitle('BSDT Sonar H7+H8a Combined Results — Odeyemi Olusegun Israel',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('h7h8_combined_results.png',dpi=150,bbox_inches='tight')
plt.show()
print('Saved: h7h8_combined_results.png')